# Fixed-Hamming-weight amplitude encoding with g-sim

This notebook demonstrates the fixed-Hamming-weight (HW) amplitude-encoding example used in the manuscript.

The goal is to prepare a state in the \(k\)-excitation subspace
\[
\mathcal H_k \subset (\mathbb C^2)^{\otimes n}, \qquad d_k=\binom{n}{k},
\]
whose amplitudes reproduce a discretized \(q\)-Gaussian target distribution. In the present experiment we use the same choice as in the manuscript:
\[
p(x) \propto (1+x^2)^{-2}, \qquad x \in [-2,2],
\]
sampled on \(d_k\) grid points.

The encoder is built from a sequence of (controlled) reconfigurable beam-splitter (RBS) gates. Rather than simulating the full Hilbert-space evolution, `g-sim` propagates the corresponding projector in the MGGM basis of the fixed-HW sector and then reads out the diagonal coordinates as probabilities.

Because this is a deterministic state-preparation task rather than a variational optimization problem, the notebook is organized a little differently from the peQNN and TFIM tutorials:

1. define a small demonstration instance;
2. inspect the fixed-HW ordering and target distribution;
3. run the `g-sim` encoder;
4. compare against a direct state-vector reference simulation;
5. plot the encoded probabilities against the target distribution.

The small reference problem is chosen so that the full \(2^n\)-dimensional state-vector simulation is still cheap. The same `g-sim` routine can then be used at much larger \(n\), where direct simulation becomes impractical.

## 1. Imports and helper modules

The notebook assumes that the experiment helpers live next to this notebook and that the `gsim` package has been installed in editable mode, for example with

```bash
python -m pip install -e .
```

from the repository root.

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

from hwencoder_helpers import build_state_maps, get_circuit_gates
from hwencoder_simulator import simulator

## 2. Define a small demonstration problem

For the manuscript experiment, the physically interesting regime is large \(n\) with fixed \(k=2\), because then
\[
d_k=\binom{n}{2}
\]
still grows polynomially while the full Hilbert space grows exponentially. For a tutorial, however, we first choose a small instance so that a direct reference simulation remains simple.

The target amplitudes are built from the normalized probabilities
\[
p_j \propto (1+x_j^2)^{-2}, \qquad x_j=-2+\frac{4j}{d_k-1}, \qquad j=0,\dots,d_k-1,
\]
and the deterministic encoder angles are the hyperspherical coordinates
\[
\theta_j=\operatorname{atan2}\!\left(\sqrt{\sum_{j'>j} a_{j'}^2},\, a_j\right),
\]
with \(a_j=\sqrt{p_j}\), and the obvious last-step specialization.

In [ ]:
# Small demonstration instance
N = 6
K = 2

# Basis/order bookkeeping
chase_states, logical_states, state_to_logical, chase_to_logical = build_state_maps(N, K)
gates = get_circuit_gates(chase_states)
DK = math.comb(N, K)

print(f"n = {N}")
print(f"k = {K}")
print(f"d_k = binom(n, k) = {DK}")
print(f"number of encoder gates = {len(gates)}")

print("\nFirst few Chase-ordered states:")
for idx, st in enumerate(chase_states[:min(8, len(chase_states))]):
    print(f"{idx:2d}: {st}")

print("\nFirst few logical (lexicographic) states:")
for idx, st in enumerate(logical_states[:min(8, len(logical_states))]):
    print(f"{idx:2d}: {st}")

## 3. Build the target distribution explicitly

Before calling the simulator, it is useful to inspect the target distribution and the resulting deterministic rotation angles. The `hwencoder_simulator.simulator(...)` function will do these steps internally as well, but spelling them out here makes the structure of the state-preparation problem transparent.

In [ ]:
# Target distribution on d_k grid points
xs = np.array([-2 + 4 * j / (DK - 1) for j in range(DK)], dtype=np.float64)
ps_target = np.array([(1 + x**2) ** (-2) for x in xs], dtype=np.float64)
ps_target = ps_target / ps_target.sum()
amps_target = np.sqrt(ps_target)

# Deterministic hyperspherical angles
thetas = []
for j in range(DK - 2):
    rem_norm = np.sqrt(np.sum(amps_target[j + 1:] ** 2))
    thetas.append(math.atan2(rem_norm, amps_target[j]))
thetas.append(math.atan2(amps_target[-1], amps_target[-2]))
thetas = np.array(thetas, dtype=float)

print("First few grid points x_j:")
print(xs[:8])

print("\nFirst few target probabilities:")
print(ps_target[:8])

print("\nFirst few encoder angles:")
print(thetas[:8])

print("\nNormalization checks:")
print("sum(p_target) =", ps_target.sum())
print("sum(a_target^2) =", np.sum(amps_target**2))

## 4. Run the `g`-sim encoder

The helper `simulator(n, k)` constructs the same target distribution, builds the Chase-ordered sequence of controlled RBS gates, maps each of those gates to a sparse MGGM generator, and propagates the initial projector in adjoint space.

The output `ps_sim` is then obtained by reading out the diagonal MGGM coordinates corresponding to the Chase-ordered computational basis states.

In [ ]:
xs_gsim, ps_gsim, ps_target_gsim = simulator(n=N, k=K)

print("g-sim probability vector (first few entries):")
print(ps_gsim[:8])

print("\nTarget probability vector from simulator (first few entries):")
print(ps_target_gsim[:8])

print("\nDo both target constructions agree?")
print(np.allclose(xs, xs_gsim), np.allclose(ps_target, ps_target_gsim))

## 5. Direct state-vector reference simulation

To validate the reduced-space result, we now simulate the same encoder directly on the full \(2^n\)-dimensional Hilbert space.

The key point is that each Chase step specifies one controlled RBS gate. On the relevant two-dimensional subspace spanned by \(|10\rangle\) and \(|01\rangle\), we use the real rotation
\[
|10\rangle \mapsto \cos(\theta)\,|10\rangle + \sin(\theta)\,|01\rangle,\qquad
|01\rangle \mapsto -\sin(\theta)\,|10\rangle + \cos(\theta)\,|01\rangle,
\]
conditioned on the control qubits being occupied. For the small system size chosen here, applying these gates directly to the full state vector is still inexpensive.

In [ ]:
# --- Direct reference simulation on the full 2^n Hilbert space ---

def bitstring_to_index(bitstring):
    # Qubit 0 is treated as the leftmost bit.
    out = 0
    for b in bitstring:
        out = (out << 1) | int(b)
    return out

def get_bit(index, n, q):
    # Extract qubit-q bit with q counted from the left.
    return (index >> (n - 1 - q)) & 1

def apply_controlled_rbs(state, n, in_idx, out_idx, ctrl, theta):
    new_state = state.copy()
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    for idx in range(2**n):
        if get_bit(idx, n, in_idx) != 1:
            continue
        if get_bit(idx, n, out_idx) != 0:
            continue
        if any(get_bit(idx, n, c) != 1 for c in ctrl):
            continue

        partner = idx ^ (1 << (n - 1 - in_idx)) ^ (1 << (n - 1 - out_idx))

        amp_a = state[idx]      # |...10...>
        amp_b = state[partner]  # |...01...>

        new_state[idx] = cos_t * amp_a - sin_t * amp_b
        new_state[partner] = sin_t * amp_a + cos_t * amp_b

    return new_state

def run_reference_encoder(n, k):
    chase_states, logical_states, state_to_logical, chase_to_logical = build_state_maps(n, k)
    gates = get_circuit_gates(chase_states)
    dk = math.comb(n, k)

    xs = np.array([-2 + 4 * j / (dk - 1) for j in range(dk)], dtype=np.float64)
    ps_target = np.array([(1 + x**2) ** (-2) for x in xs], dtype=np.float64)
    ps_target = ps_target / ps_target.sum()
    amps_target = np.sqrt(ps_target)

    thetas = []
    for j in range(dk - 2):
        rem_norm = np.sqrt(np.sum(amps_target[j + 1:] ** 2))
        thetas.append(math.atan2(rem_norm, amps_target[j]))
    thetas.append(math.atan2(amps_target[-1], amps_target[-2]))

    # Start in the first Chase basis state
    psi = np.zeros(2**n, dtype=complex)
    psi[bitstring_to_index(chase_states[0])] = 1.0

    for theta, (in_idx, out_idx, ctrl) in zip(thetas, gates):
        psi = apply_controlled_rbs(psi, n, in_idx, out_idx, ctrl, theta)

    probs = np.array([abs(psi[bitstring_to_index(st)])**2 for st in chase_states], dtype=float)
    return xs, probs, ps_target, psi

xs_ref, ps_ref, ps_target_ref, psi_ref = run_reference_encoder(N, K)

## 6. Pointwise verification

The direct reference simulator and the reduced adjoint-space simulator should reproduce the same final probability vector up to numerical precision.

In [ ]:
print("sum(p_gsim)   =", np.sum(ps_gsim))
print("sum(p_ref)    =", np.sum(ps_ref))
print("sum(p_target) =", np.sum(ps_target))

print("\nmax abs. error (g-sim vs reference):", np.max(np.abs(ps_gsim - ps_ref)))
print("mean abs. error (g-sim vs reference):", np.mean(np.abs(ps_gsim - ps_ref)))

print("\nmax abs. error (g-sim vs target):", np.max(np.abs(ps_gsim - ps_target)))
print("mean rel. error (g-sim vs target):", np.mean(np.abs(ps_gsim - ps_target) / np.maximum(ps_target, 1e-15)))

print("\nallclose(g-sim, reference)?", np.allclose(ps_gsim, ps_ref))

## 7. Final comparison plot

The plot below compares three objects:

- the target discretized \(q\)-Gaussian probabilities,
- the probabilities obtained from the reduced `g-sim` simulation,
- the probabilities obtained from the direct full state-vector reference simulation.

For this small validation instance, the `g-sim` and reference curves should sit on top of each other. In larger-system runs, one would typically keep only the `g-sim` curve and compare it against the target distribution.

In [ ]:
plt.figure(figsize=(6.4, 3.9))
plt.scatter(xs_gsim, ps_gsim, label="g-sim", alpha=0.85)
plt.scatter(xs_ref, ps_ref, marker="x", label="direct state-vector reference", alpha=0.85)
plt.plot(xs, ps_target, "--", linewidth=1.4, label="target distribution")
plt.xlabel(r"$x$")
plt.ylabel("probability")
plt.title("Fixed-HW amplitude encoding of a discretized q-Gaussian")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

## 8. Remarks

- The notebook validates the fixed-HW encoder on a deliberately small instance where the full Hilbert-space simulation remains cheap.
- The `g-sim` simulation itself, however, only evolves the problem inside the reduced MGGM description of the fixed-\(k\) sector.
- For the manuscript-scale experiment, one keeps \(k=2\) fixed and increases \(n\), so that
  \[
  d_k=\binom{n}{2}
  \]
  grows only quadratically, while the full Hilbert-space dimension \(2^n\) grows exponentially.
- In that regime, the same helper `simulator(n, k)` can still generate the encoded probability distribution, whereas the direct reference simulation used here quickly becomes impractical.

As a next step, you can simply increase `N` in the earlier cells and keep `K = 2` fixed to move from this small verification setting toward the large-scale state-preparation regime highlighted in the manuscript.